# Updating a PMO with new metadata 

Creating a PMO from the data downloaded from the SRA associated with the following study and updating with meta data:

Furstenau, T. N., Whealy, R., Timm, S., Roberts, A., Maltinsky, S., Wells, S. J., Drake, K., Ross, A., Bolduc, C., Pearson, T., & Fofanov, V. Y. (2025). *High-throughput targeted amplicon screening tool for characterizing intrahost diversity in* Staphylococcus aureus *directly from sample*. Microbial Genomics, 11(6). https://doi.org/10.1099/mgen.0.001427

This will cover how to update specimen meta after building a minimum PMO

In [16]:
import pandas as pd
from pmotools.pmo_engine.pmo_writer import * 
from pmotools.pmo_engine.pmo_reader import PMOReader
from pmotools.pmo_builder.metatable_to_pmo import specimen_info_table_to_pmo
from pmotools.pmo_builder.pmo_updater import PMOUpdater


## Read in data

See [Create minimal pmo](./Create_minimal_pmo.ipynb) page for creating a minimum PMO that will be read below

In [24]:
staph_aureus_pmo = PMOReader.read_in_pmo("minimum_Furstenau2025_new_names_PMO.json.gz")


### Adding in specimen meta information 

Right now the specimen_info is simply the specimen_name. Additional info can be added 


In [25]:
staph_aureus_pmo["specimen_info"][0]

{'specimen_name': '85b498-Wk16-Nasal'}

First use `specimen_info_table_to_pmo` and then merge into the specimen_info already present. There are several columns in the SRA/ENA metadata but for now we use as example the geographic location `country` and the date of collection `collection_date`.



In [26]:
sra_info = pd.read_csv("sra_info_table.tsv", sep = '\t')
sra_meta_of_interest = sra_info[['sample_alias', 'country', 'collection_date']].drop_duplicates()
sra_meta_of_interest.head()

,sample_alias,country,collection_date
0,2b2068n1,USA: Arizona,2019
2,2a4023n1,USA: Arizona,2019
3,2b4022n1,USA: Arizona,2019
5,2b3034n1,USA: Arizona,2019
9,2b2048n1,USA: Arizona,2019


We will want a field for country only and then state so will create new columns by split on ":" 

In [27]:
sra_meta_of_interest[['country_only', 'state']] = sra_meta_of_interest['country'].str.split(':', expand=True)
sra_meta_of_interest.head()

,sample_alias,country,collection_date,country_only,state
0,2b2068n1,USA: Arizona,2019,USA,Arizona
2,2a4023n1,USA: Arizona,2019,USA,Arizona
3,2b4022n1,USA: Arizona,2019,USA,Arizona
5,2b3034n1,USA: Arizona,2019,USA,Arizona
9,2b2048n1,USA: Arizona,2019,USA,Arizona


Now build the specimen meta 

In [28]:
pmo_spec_info = specimen_info_table_to_pmo(
                            sra_meta_of_interest, 
                            specimen_name_col='sample_alias',
                            collection_date_col='collection_date',
                            collection_country_col='country_only',
                            geo_admin1_col='state',
                           )

Now join this into the specimen_info using `PMOUpdater.merge_dicts_by_key`

In [29]:
from pmotools.pmo_builder.pmo_updater import PMOUpdater

staph_aureus_pmo["specimen_info"] = PMOUpdater.merge_dicts_by_key(
    staph_aureus_pmo["specimen_info"],
    pmo_spec_info, 
    key_field="specimen_name")

Now the specimen_info has meta data 

In [30]:
staph_aureus_pmo["specimen_info"][0]

{'specimen_name': '85b498-Wk16-Nasal',
 'collection_date': '2022-02-07',
 'collection_country': 'USA',
 'geo_admin1': ' Phoenix'}

Now let's write and validate the pmo with meta 

In [31]:
pmowriter = PMOWriter()
pmowriter.write_out_pmo(staph_aureus_pmo, "minimum_Furstenau2025_PMO_with_spec_meta.json.gz", overwrite=True)

In [32]:
!pmotools-python validate_pmo --pmo minimum_Furstenau2025_PMO_with_spec_meta.json.gz --jsonschema_version 1.1.0